# Two-Tower Shared-State Baseline

The previous experiments showed that the NLI model jointly encodes the
state and hypothesis, preventing straightforward reuse of the final state
representation.

This experiment tests a simpler two-tower architecture:

1. Encode the state once.
2. Encode all questions independently in a batch.
3. Compare the state embedding with each question embedding.
4. Use similarity as a first approximation of decision relevance.

No training or calibration is performed at this stage.

In [1]:
from sentence_transformers import SentenceTransformer

In [2]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda",
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
state = """
Inspect the repository, identify which commit introduced the login bug,
run the relevant tests, and explain the likely cause.
"""

In [4]:
hypotheses = [
    "This task requires filesystem access.",
    "This task requires Git access.",
    "This task requires shell execution.",
    "This task requires browser access.",
    "This task requires email access.",
    "This task requires calendar access.",
]

In [5]:
state_embedding = model.encode(
    state,
    convert_to_tensor=True,
)

In [6]:
question_embeddings = model.encode(
    hypotheses,
    convert_to_tensor=True,
)

In [7]:
print("State:", state_embedding.shape)
print("Questions:", question_embeddings.shape)

State: torch.Size([384])
Questions: torch.Size([6, 384])


In [8]:
from sentence_transformers import util

In [9]:
similarities = util.cos_sim(
    state_embedding,
    question_embeddings,
)[0]

similarities

tensor([0.0845, 0.2250, 0.0401, 0.0948, 0.1094, 0.0633], device='cuda:0')

In [10]:
for hypothesis, score in zip(hypotheses, similarities):
    print(f"{score.item():.4f}  {hypothesis}")

0.0845  This task requires filesystem access.
0.2250  This task requires Git access.
0.0401  This task requires shell execution.
0.0948  This task requires browser access.
0.1094  This task requires email access.
0.0633  This task requires calendar access.


## Similarity Scores

The two-tower model produces cosine similarity scores between the shared
state embedding and each question embedding.

These scores are **not calibrated probabilities**. They only measure
semantic similarity in the embedding space.

For now, this experiment tests whether independently encoded questions
can still capture useful routing signals while allowing the state to be
encoded only once.

## Comparison with the NLI Baseline

The two-tower model provides genuine shared-state encoding, but cosine
similarity alone may not be sufficient for decision routing.

Next, we compare its scores with the existing NLI backend using the same
state and hypotheses.

In [11]:
from parallel_decider import BooleanQuestion, NLIBackend

nli_backend = NLIBackend()

/home/mebus/workspace/parallel-decider/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [12]:
questions = [
    BooleanQuestion(
        name=f"q_{index}",
        hypothesis=hypothesis,
    )
    for index, hypothesis in enumerate(hypotheses)
]

In [13]:
nli_results = nli_backend.decide(
    state=state,
    questions=questions,
)

In [14]:
import pandas as pd

comparison = pd.DataFrame(
    {
        "hypothesis": hypotheses,
        "two_tower_similarity": [
            score.item()
            for score in similarities
        ],
        "nli_probability": [
            result.probability
            for result in nli_results
        ],
    }
)

comparison

,hypothesis,two_tower_similarity,nli_probability
0,This task requires filesystem access.,0.084494,0.929199
1,This task requires Git access.,0.225041,0.921387
2,This task requires shell execution.,0.040054,0.070129
3,This task requires browser access.,0.094790,0.019150
4,This task requires email access.,0.109350,0.001127
5,This task requires calendar access.,0.063290,0.000121


In [15]:
comparison.sort_values(
    "nli_probability",
    ascending=False,
)

,hypothesis,two_tower_similarity,nli_probability
0,This task requires filesystem access.,0.084494,0.929199
1,This task requires Git access.,0.225041,0.921387
2,This task requires shell execution.,0.040054,0.070129
3,This task requires browser access.,0.094790,0.019150
4,This task requires email access.,0.109350,0.001127
5,This task requires calendar access.,0.063290,0.000121


## Asymmetric Two-Tower Baseline

The first two-tower experiment used a general semantic-similarity model.
While it successfully identified Git as relevant, its ranking of the other
capabilities was poor compared with the NLI baseline.

Next, we test an asymmetric retrieval model. The user task is encoded as a
query, while each routing hypothesis is encoded as a candidate passage.

This still preserves the main architectural property we are interested in:
the state is encoded only once.

In [16]:
from sentence_transformers import SentenceTransformer, util

e5_model = SentenceTransformer(
    "intfloat/e5-small-v2",
    device="cuda",
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [17]:
query = f"query: {state}"

passages = [
    f"passage: {hypothesis}"
    for hypothesis in hypotheses
]

In [18]:
e5_state_embedding = e5_model.encode(
    query,
    convert_to_tensor=True,
    normalize_embeddings=True,
)

e5_question_embeddings = e5_model.encode(
    passages,
    convert_to_tensor=True,
    normalize_embeddings=True,
)

In [19]:
e5_similarities = util.cos_sim(
    e5_state_embedding,
    e5_question_embeddings,
)[0]

In [20]:
for hypothesis, score in zip(
    hypotheses,
    e5_similarities,
):
    print(
        f"{score.item():.4f}  "
        f"{hypothesis}"
    )

0.7870  This task requires filesystem access.
0.7977  This task requires Git access.
0.7938  This task requires shell execution.
0.8075  This task requires browser access.
0.7843  This task requires email access.
0.7749  This task requires calendar access.


In [21]:
comparison["e5_similarity"] = [
    score.item()
    for score in e5_similarities
]

comparison

,hypothesis,two_tower_similarity,nli_probability,e5_similarity
0,This task requires filesystem access.,0.084494,0.929199,0.786985
1,This task requires Git access.,0.225041,0.921387,0.797686
2,This task requires shell execution.,0.040054,0.070129,0.793829
3,This task requires browser access.,0.094790,0.019150,0.807467
4,This task requires email access.,0.109350,0.001127,0.784279
5,This task requires calendar access.,0.063290,0.000121,0.774895


## Observation

The asymmetric E5 retrieval model does not improve the decision signal.

All routing hypotheses receive similarly high cosine-similarity scores,
including clearly irrelevant capabilities such as browser, email, and calendar.

This suggests that retrieval-style semantic similarity is not sufficient for
our task. The desired behavior is closer to natural-language inference:
given a state and a proposition, determine whether the proposition is supported
by the state.

The shared-state architecture therefore likely needs a learned decision scorer,
not raw embedding similarity.

In [22]:
from parallel_decider.two_tower import TwoTowerDecisionHead
import torch

head = TwoTowerDecisionHead(
    embedding_dim=384,
    hidden_dim=256,
).to("cuda")

In [23]:
logits = head(
    state_embedding,
    question_embeddings,
)

probabilities = torch.sigmoid(logits)

probabilities

tensor([0.5056, 0.5048, 0.5093, 0.5066, 0.5067, 0.5041], device='cuda:0',
       grad_fn=<SigmoidBackward0>)

In [24]:
for hypothesis, probability in zip(
    hypotheses,
    probabilities,
):
    print(
        f"{probability.item():.4f}  "
        f"{hypothesis}"
    )

0.5056  This task requires filesystem access.
0.5048  This task requires Git access.
0.5093  This task requires shell execution.
0.5066  This task requires browser access.
0.5067  This task requires email access.
0.5041  This task requires calendar access.
